### Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [ ]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen2 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.97G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/267 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

### Data Prep

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ikram98ai/compliance_verification", split = "train")

README.md:   0%|          | 0.00/469 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/995k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/248k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15587 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3897 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['image_urls', 'compliance_status', 'violation_reason'],
    num_rows: 15587
})

In [ ]:
dataset[2]["image_urls"]

['https://cf.freshprints.com/designs/1694998809572oaoms_nt_front.png',
 'https://cf.freshprints.com/designs/1694998809572abohc_nt_back.png']

In [ ]:
dataset[2]["compliance_status"], dataset[2]["violation_reason"]

('Non-compliant',
 'This design is rejected. Needs to be all 3 greek letters.We welcome you to modify and resubmit the design for further review.')

In [ ]:
system_prompt = """You are a licensing compliance expert specifically for university and Greek organization apparel.
Your task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine
if a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place.
For each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or
'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief
explanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations
present in the image, not hypothetical concerns."""

instruction = """Review this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."""


def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
        "content" : [
            {"type" : "text",  "text"  : system_prompt + "\n\n" + instruction},
            ] + [{"type" : "image", "image" : img_url} for img_url in sample["image_urls"]]
        },
        { "role" : "assistant",
        "content" : [
            {"type" : "text",  "text"  : f"Compliance Status: {sample['compliance_status']}\nViolation Reason: {sample['violation_reason']}"} ]
        },
    ]
    return { "messages" : conversation }



Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

We look at how the conversations are structured for the first example:

In [ ]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': "You are a licensing compliance expert specifically for university and Greek organization apparel.\nYour task is to evaluate designs against the established licensing guidelines of these specific organizations. Determine\nif a design meets all requirements or violates any rules, assuming proper licensing permissions are already in place.\nFor each evaluation, you must respond in a strict two-line format: first indicating 'Compliance Status: Compliant' or\n'Compliance Status: Non-compliant', followed by 'Violation Reason:' with either 'None' for compliant designs or a brief\nexplanation for non-compliant designs. Never elaborate beyond this format. Base your evaluation solely on actual violations\npresent in the image, not hypothetical concerns.\n\nReview this apparel design for compliance with licensing rules. Provide compliance status and violation reason, if any."},
    {'type': 'image',
     'image': 'https:

In [ ]:
import requests
from PIL import Image as PILImage
from io import BytesIO

def load_image_from_url(url):
    """Helper function to download and convert image from URL"""
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        return PILImage.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Error loading image from {url}: {str(e)}")
        return None

In [ ]:
idx = 300
dataset[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1710731836981mjudw_nt_front.png',
  'https://cf.freshprints.com/designs/1710731836981yantr_nt_back.png'],
 'compliance_status': 'Compliant',
 'violation_reason': 'None'}

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image_urls = dataset[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Compliance Status: Compliant  
Violation Reason: None<|im_end|>


In [ ]:
# generated_text = model.generate(**inputs,max_new_tokens = 128, use_cache = True, temperature = 1.5, min_p = 0.1)
# tokenizer.decode(generated_text[0]).split("|im_start|>assistant\n")[-1].split("<|im_end")[0]

### Train the model

In [ ]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4 ,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
7.854 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,587 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 103,043,072/7,000,000,000 (1.47% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,7.592300
2,7.578700
3,7.022300
4,4.333200
5,3.073500
6,2.595100
7,2.100900
8,1.631600
9,1.148900
10,0.809700


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2560.818 seconds used for training.
42.68 minutes used for training.
Peak reserved memory = 17.025 GB.
Peak reserved memory for training = 9.171 GB.
Peak reserved memory % of max memory = 76.824 %.
Peak reserved memory for training % of max memory = 41.384 %.


### Inference

In [ ]:
idx = 7900
dataset[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1694660499473ytvak_nt_front.png',
  'https://cf.freshprints.com/designs/1694660499473fryqd_nt_back.png'],
 'compliance_status': 'Non-compliant',
 'violation_reason': 'This design is rejected. Please use a capital A and make the D and P more distinct so it is clear who this is representing.We welcome you to modify and resubmit the design for further review.'}

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image_urls = dataset[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 64,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Compliance Status: Compliant
Violation Reason: None<|im_end|>


### Saving, loading finetuned Lora adapter

In [ ]:
# model.save_pretrained("compliance_verification_lora_model")
# tokenizer.save_pretrained("compliance_verification_lora_model")
from google.colab import userdata

model.push_to_hub("ikram98ai/compliance_verification_lora_model",token= userdata.get('HF_TOKEN'))
tokenizer.push_to_hub("ikram98ai/compliance_verification_lora_model", token= userdata.get('HF_TOKEN'))

README.md:   0%|          | 0.00/610 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Saved model to https://huggingface.co/ikram98ai/compliance_verification_lora_model


  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
idx = 10000
dataset[idx]

{'image_urls': ['https://cf.freshprints.com/designs/1732223634245ynhmq_nt_front.png'],
 'compliance_status': 'Non-compliant',
 'violation_reason': 'This design is rejected. Phrase is not approved.We welcome you to modify and resubmit the design for further review.'}

In [ ]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "lora_model",
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

image_urls = dataset[idx]["image_urls"]

messages = [
    {"role": "user", "content": [ {"type": "image"} for url in image_urls ] + [
        {"type": "text", "text": system_prompt + "\n\n" + instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    [load_image_from_url(url) for url in image_urls],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Compliance Status: Non-compliant
Violation Reason: This design is rejected. Please use the official name of our organization.We welcome you to modify and resubmit the design for further review.<|im_end|>


In [ ]:
# To export and save to your Hugging Face account
if True: model.push_to_hub_merged("ikram98ai/compliance_verification_model", tokenizer, token= userdata.get('HF_TOKEN'))

  0%|          | 0/1 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit:   0%|          | 0/5 [00:00<?, ?it/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/3.90G [00:00<?, ?B/s]

...

Unsloth: Merging weights into 16bit:  80%|████████  | 4/5 [16:23<04:49, 289.94s/it]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 5/5 [18:07<00:00, 217.45s/it]
